# 6. Exploring Ongoing Research and Development in Causal Model Parameterization in pgmpy

### 6.1. Challenges

#### 6.1.1. Real-world data come in many forms.

pgmpy currently provides models such as `DiscreteBayesianNetwork`, `LinearGaussianBayesianNetwork`, `FunctionalBayesianNetwork`, and `DynamicBayesianNetwork`.

However, real-world data come in many different forms.

![006](../image/06-001-EN.excalidraw.png)

#### 6.1.2. Representing discrete distributions can sometimes cause memory usage to explode.

> Real-world Bayesian networks commonly have high-cardinality nodes. <br>
> Issue #1776 shows another user hit 64 TiB allocation on an 82-node network. <br>
> This is silently killing pgmpy adoption in production systems. [[#3203](https://github.com/pgmpy/pgmpy/issues/3203)]


### 6.2. Proposed Direction

#### 6.2.1. Enable individual parameter learning to represent each parameter effectively.

> [Bayesian networks with a logistic regression model for the conditional probabilities](https://www.sciencedirect.com/science/article/pii/S0888613X08000121)

The paper's idea: Replace conditional probability tables (CPTs) with `LogisticRegression` models.

#### 6.2.2. Extend this idea to machine learning and deep learning.

-> Ensure that each pgmpy parameter is compatible with the `scikit-learn` ecosystem, thereby increasing the expressive power of individual parameters.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from skpro.regression.linear import GLMRegressor

from pgmpy_tutorial.base._base import _CoreGraph as BayesianNetwork
from pgmpy_tutorial.parameter import LinearGaussianCPD, SklearnAdapter, SkproAdapter, TabularCPD
from pgmpy_tutorial.refactored_inference.LW import LikelihoodWeighting

np.random.default_rng(42)


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n_samples = 10_000

A = rng.binomial(
    n=1,
    p=0.35,
    size=n_samples,
)


# ------------------------------------------------------------
# C ~ Normal(0, 1)
# ------------------------------------------------------------

C = rng.normal(
    loc=0.0,
    scale=1.0,
    size=n_samples,
)


# ------------------------------------------------------------
# B | A ~ Normal(1 + 2.5 A, 0.5^2)
#
# A=0: B ~ Normal(1.0, 0.5^2)
# A=1: B ~ Normal(3.5, 0.5^2)
# ------------------------------------------------------------

B = (
    1.0
    + 2.5 * A
    + rng.normal(
        loc=0.0,
        scale=0.5,
        size=n_samples,
    )
)


# ------------------------------------------------------------
# P(D=1 | C) = sigmoid(-0.4 + 1.3 C)
# ------------------------------------------------------------

logit_D = -0.4 + 1.3 * C
prob_D = 1.0 / (1.0 + np.exp(-logit_D))

D = rng.binomial(
    n=1,
    p=prob_D,
    size=n_samples,
)


# ------------------------------------------------------------
# E | B,C ~ Normal(2 + 1.2 B - 0.8 C, 0.7^2)
# ------------------------------------------------------------

E = (
    2.0
    + 1.2 * B
    - 0.8 * C
    + rng.normal(
        loc=0.0,
        scale=0.7,
        size=n_samples,
    )
)


data = pd.DataFrame(
    {
        "A": A,
        "B": B,
        "C": C,
        "D": D,
        "E": E,
    }
)

print(data.head())
print(data.dtypes)


In [ ]:
bn = BayesianNetwork()
bn.SUPPORTED_EDGE_TYPES = frozenset(["->", "<-"])  # DAG

bn.add_edges_from(
    [
        ("A", "B", "->"),
        ("C", "D", "->"),
        ("B", "E", "->"),
        ("C", "E", "->"),
    ]
)

cpd_A = TabularCPD(categories={"A": [0, 1]})

cpd_B = SklearnAdapter(LinearRegression())

cpd_C = LinearGaussianCPD()

cpd_D = SklearnAdapter(
    LogisticRegression(
        max_iter=1_000,
        random_state=42,
    )
)

cpd_E = SkproAdapter(
    GLMRegressor(
        family="Normal",
        link="Identity",
        add_constant=True,
    )
)
        

In [ ]:
bn.add_cpd("A", cpd_A)
bn.add_cpd("B", cpd_B)
bn.add_cpd("C", cpd_C)
bn.add_cpd("D", cpd_D)
bn.add_cpd("E", cpd_E)


In [ ]:
# P(A)
cpd_A.fit(data[["A"]])

# P(B | A)
cpd_B.fit(
    X=data[["A"]],
    y=data[["B"]],
)

# P(C)
cpd_C.fit(data[["C"]])

# P(D | C)
cpd_D.fit(
    X=data[["C"]],
    y=data[["D"]],
)

# P(E | B, C)
cpd_E.fit(
    X=data[["B", "C"]],
    y=data["E"],
)


In [ ]:
pgm = bn.to_daft()
pgm.render()


In [ ]:
# n_samples=1000
inference = LikelihoodWeighting()

posterior = inference.query(
    bn,
    variables=["A"],
    evidence={
        "B": 2.2,
    },
    do={"E": 5},
    n_samples=200,
    seed=42,
)


In [ ]:
posterior
